In [16]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import combinations
from sklearn.model_selection import train_test_split

import heapq
from random import randrange
from random import seed as set_seed
import numpy as np
#from numba import njit, prange
from pandas.api.types import is_numeric_dtype

import numpy as np
import pandas as pd
import torch.utils.data

from scipy.sparse.linalg import svds

In [17]:
class MovieLens20MDataset(torch.utils.data.Dataset):
    """
    MovieLens 20M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path, sep=',', engine='c', header='infer'):
        self.data = pd.read_csv(dataset_path, sep=sep, engine=engine, header=header).to_numpy()[:, :4]
        self.items = self.data[:, :2].astype(np.int32) - 1  # -1 because ID begins from 1
        self.targets = self.__preprocess_target(self.data[:, 2]).astype(np.float32)
        self.field_dims = np.max(self.items, axis=0) + 1
        self.user_field_idx = np.array((0, ), dtype=np.int64)
        self.item_field_idx = np.array((1,), dtype=np.int64)

    def __len__(self):
        return self.targets.shape[0]

    def __getitem__(self, index):
        return self.items[index], self.targets[index]

    def __preprocess_target(self, target):
        target[target <= 0] = 0
        target[target > 0] = 1
        return target


class MovieLens1MDataset(MovieLens20MDataset):
    """
    MovieLens 1M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path):
        super().__init__(dataset_path, sep='::', engine='python', header=None)

In [18]:
dataset_path = "ratings.dat"
dataset = MovieLens1MDataset(dataset_path)

In [19]:
user_num = dataset.field_dims[0]
item_num = dataset.field_dims[1]
print("Number of users: ", user_num, ", Number of items: ", item_num)

Number of users:  6040 , Number of items:  3952


In [20]:
columns_name=['user_id','item_id','rating', 'timestamp']
df = pd.DataFrame(dataset.data, columns = columns_name)
df

,user_id,item_id,rating,timestamp
0,1,1193,1,978300760
1,1,661,1,978302109
2,1,914,1,978301968
3,1,3408,1,978300275
4,1,2355,1,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,1,956704887
1000206,6040,562,1,956704746
1000207,6040,1096,1,956715648


In [21]:
df_sorted = df.sort_values(by='timestamp')

test_treshold = int(len(df_sorted) * 0.98)
val_treshold = int(len(df_sorted) * 0.96)

train_val = df_sorted.head(test_treshold)
warm_test = df_sorted.tail(len(df_sorted) - test_treshold)
test = warm_test.loc[warm_test.groupby('user_id')['timestamp'].idxmax()]
warm_t = warm_test[~warm_test.index.isin(test.index)]


train = df_sorted.head(val_treshold)
test_val = df_sorted.tail(len(df_sorted) - val_treshold)
warm_val = test_val[~test_val.index.isin(warm_test.index)]
val = warm_val.loc[warm_val.groupby('user_id')['timestamp'].idxmax()]
warm_v = warm_val[~warm_val.index.isin(val.index)]


#train = pd.concat([train, warm_v])

In [22]:
print("Train len: ", len(train))
print("Train users: ", len(train["user_id"].unique()))
print("Val len: ", len(val))
print("Val users: ", len(val["user_id"].unique()))
print("Test len: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Warm val len: ", len(warm_v))
print("Warm val users: ", len(warm_v["user_id"].unique()))
print("Warm test len: ", len(warm_t))
print("Warm test users: ", len(warm_t["user_id"].unique()))
print("test_val intersection users: ", np.intersect1d(val["user_id"].unique(), test["user_id"].unique()).shape[0])

Train len:  960200
Train users:  6037
Val len:  572
Val users:  572
Test len:  474
Test users:  474
Warm val len:  19432
Warm val users:  515
Warm test len:  19531
Warm test users:  451
test_val intersection users:  338


Сохраним все train val test в виде txt файлов будем их хранить как последовательность. в начале user_id а затем item_ids в нужном порядке.

In [23]:
user_items = train.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/MovieLens1M/train.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [11]:
user_items = warm_v.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/MovieLens1M/warm_val.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [13]:
filtered_train = warm_v[warm_v['user_id'].isin(val['user_id'])]
train_val = pd.concat([filtered_train, val], ignore_index=True)

user_items = train_val.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/MovieLens1M/val.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")


In [14]:
unique_indices = val["user_id"].unique()
with open('data2/MovieLens1M/val_users.txt', 'w') as f:
    for index in unique_indices:
        f.write(f"{index}\n")

# Чтение чисел из файла и сохранение их в список
with open('data2/MovieLens1M/val_users.txt', 'r') as f:
    index_list = [int(line.strip()) for line in f]

In [15]:
filtered_train = df_sorted[df_sorted['user_id'].isin(test['user_id'])]


user_items = filtered_train.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/MovieLens1M/test.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")
